In [ ]:
import sys
from pathlib import Path

import matplotlib_fontja  # noqa: F401

for path in [Path.cwd(), *Path.cwd().parents]:
    if (path / "common" / "notebook_style.py").exists():
        sys.path.insert(0, str(path))
        break

    if (path / "src" / "common" / "notebook_style.py").exists():
        sys.path.insert(0, str(path / "src"))
        break

    if (path / "doc" / "src" / "common" / "notebook_style.py").exists():
        sys.path.insert(0, str(path / "doc" / "src"))
        break


from common.notebook_style import apply_dataframe_style

apply_dataframe_style()

In [ ]:
# CI では短い動作確認に切り替える（通常実行では False）
CI_SMOKE_TEST = False

# 第12章：つながりを活かして推薦する

このファイルでは、第 12 章で扱う BPR、NGCF、LightGCN の実装を `src` から呼び出し、Online Retail の購入履歴を用いた、グラフを使う推薦モデルの学習と評価を実行します。評価時はモデルを 1 回実行して全ユーザーと全アイテムの埋め込みを求め、各バッチの推薦スコアをまとめて計算します。

出典: Daqing Chen, [Online Retail](https://doi.org/10.24432/C5BW33), UCI Machine Learning Repository, 2015。ライセンス: [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/)。元データをCSV形式に変換し、本文に示す加工を行います。


## 実行用の準備

次のコードでは、第 12 章の `src/main.py` を探して呼び出せるようにします。`src` を Python がモジュールを探す場所に追加し、作業フォルダを一時的に切り替える補助関数も定義します。


In [ ]:
import argparse
import importlib.util
import os
import sys
from contextlib import contextmanager


def find_repo_paths(start: Path | None = None) -> tuple[Path, Path]:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "doc/12/src/main.py").exists():
            return candidate, candidate / "doc/12/src"
        if (candidate / "12/src/main.py").exists():
            return candidate, candidate / "12/src"
    raise FileNotFoundError("第12章の src/main.py が見つかりません。")


@contextmanager
def pushd(path: Path):
    previous = Path.cwd()
    os.chdir(path)
    try:
        yield
    finally:
        os.chdir(previous)


REPO_ROOT, SRC_DIR = find_repo_paths()
MAIN_PATH = SRC_DIR / "main.py"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print(f"REPO_ROOT = {REPO_ROOT}")
print(f"SRC_DIR   = {SRC_DIR}")


def load_chapter12_main_module():
    spec = importlib.util.spec_from_file_location(
        "chapter12_runtime_main", MAIN_PATH
    )
    if spec is None or spec.loader is None:
        raise ImportError(f"main.py をロードできません: {MAIN_PATH}")
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


def run_chapter12_main(
    model_name: str = "LightGCN", overrides: dict | None = None
):
    module = load_chapter12_main_module()
    args = argparse.Namespace(dataset="online-retail", model=model_name)
    unknown_args_dict = dict(overrides or {})
    unknown_args_dict.setdefault(
        "checkpoint_dir_name", str(SRC_DIR / "checkpoint")
    )
    with pushd(SRC_DIR):
        return module.main(args, unknown_args_dict)

## 学習設定

次のコードでは、比較するモデル名と共通の設定を指定します。通常は 300 エポック、CI の動作確認では 2 エポックだけ学習します。


In [ ]:
# 比較するモデル
MODEL_NAMES = ["BPR", "NGCF", "LightGCN"]

# 実行時に上書きする設定だけを指定する
COMMON_OVERRIDES = {
    "epochs": 2 if CI_SMOKE_TEST else 300,
    "early_stop_num": 2 if CI_SMOKE_TEST else 10,
    "train_batch_size": 1024,
    "eval_batch_size": 128,
    "show_progress": not CI_SMOKE_TEST,
}

print(MODEL_NAMES)
print(COMMON_OVERRIDES)

## 学習の実行

次のコードでは、3 つのモデルを順に学習し、検証データとテストデータの結果をまとめます。


In [ ]:
RESULTS = {}
for model_name in MODEL_NAMES:
    overrides = {
        **COMMON_OVERRIDES,
        "result_file_name_postfix": f"notebook_{model_name.lower()}",
    }
    RESULTS[model_name] = run_chapter12_main(model_name, overrides)

RESULTS

## 実行結果の見方

学習が完了すると、ログに `BEST VALID RESULT` と `TEST RESULT` が表示されます。`BEST VALID RESULT` は各モデルで、検証データの Recall@10 が最も高かった時点の指標で、早期停止と学習途中のモデルを保存する基準になります。`TEST RESULT` は、そのモデルをテストデータで評価した最終的な推薦の精度です。Recall@10 は正解アイテムをどれだけ含められたか、Precision@10 は推薦した上位 10 件のうち正解がどれだけ含まれたか、NDCG@10・MRR@10・MAP@10 は正解アイテムが上位に出ているかを評価する指標です。
